# Chapter 11 — Testing

Interviewers want to see that you can: understand the big picture, see how the pieces fit, organize tests into categories, and be practical.

Test categories: **normal cases**, **extremes** (empty, huge, negative), **nulls/illegal input**, **strange input**. Unit tests with `unittest` or `pytest`, property-based tests with `hypothesis`.

Here each question is paired with an executable test suite. We use `unittest` so the notebook needs no extra dependencies.

In [1]:
import unittest


def run(*cases: type[unittest.TestCase]) -> None:
    suite = unittest.TestSuite()
    for c in cases:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(c))
    result = unittest.TextTestRunner(verbosity=1).run(suite)
    assert result.wasSuccessful()

## 11.1 Mistake
The C code `for (unsigned int i = 100; i >= 0; --i) printf("%d", i);` loops forever. `i` is **unsigned**, so `i >= 0` is always true and `--i` wraps around to 4,294,967,295. (It also prints with `%d` instead of `%u`.) Python ints never wrap, but we can emulate the bug to show it, then test the fix.

In [2]:
def buggy_countdown(max_iters: int = 200) -> list[int]:
    out, i = [], 100
    while i >= 0 and len(out) < max_iters:   # guard so the demo terminates
        out.append(i)
        i = (i - 1) & 0xFFFFFFFF            # unsigned 32-bit decrement
    return out


def fixed_countdown() -> list[int]:
    return list(range(100, -1, -1))  # i > 0 / signed int / print 0 explicitly


class TestCountdown(unittest.TestCase):
    def test_bug_wraps_around(self):
        self.assertEqual(buggy_countdown()[101], 4294967295)

    def test_fixed(self):
        out = fixed_countdown()
        self.assertEqual((out[0], out[-1], len(out)), (100, 0, 101))


run(TestCountdown)

.

.

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


## 11.2 Random Crashes
An app crashes at a different spot each time it runs. Likely causes: randomness or time-dependent input, uninitialized variables, memory leaks, external dependencies, and **race conditions**. The fix is to make it reproducible. Log aggressively, then seed the RNGs and freeze time. Below is a flaky function and a deterministic test harness around it.

In [3]:
import random
from unittest import mock


def process(items: list[int]) -> float:
    random.shuffle(items)          # hidden nondeterminism
    return 1 / items[0]            # crashes whenever 0 lands first


class TestRandomCrash(unittest.TestCase):
    def test_reproduce_with_seed(self):
        crashes = []
        for seed in range(50):
            random.seed(seed)
            try:
                process([0, 1, 2, 3])
            except ZeroDivisionError:
                crashes.append(seed)
        self.assertTrue(crashes, "should find a crashing seed")
        random.seed(crashes[0])      # reproduce deterministically
        with self.assertRaises(ZeroDivisionError):
            process([0, 1, 2, 3])

    def test_mock_out_randomness(self):
        with mock.patch("random.shuffle", lambda xs: xs.sort(reverse=True)):
            self.assertAlmostEqual(process([0, 1, 2, 3]), 1 / 3)


run(TestRandomCrash)

.

.

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


## 11.3 Chess Test
Test a chess game's `can_move_to(x, y)` method on a piece. Test categories: **extremes** (off the board, the board's edges), **general valid/invalid moves** for each piece type, and moves **blocked** by other pieces.

In [4]:
from dataclasses import dataclass


@dataclass
class Piece:
    kind: str       # "rook", "bishop", "knight"
    x: int
    y: int

    def can_move_to(self, x: int, y: int, occupied: set[tuple[int, int]] = frozenset()) -> bool:
        if not (0 <= x < 8 and 0 <= y < 8) or (x, y) == (self.x, self.y):
            return False
        dx, dy = x - self.x, y - self.y
        match self.kind:
            case "knight":
                return {abs(dx), abs(dy)} == {1, 2}
            case "rook" if dx and dy:
                return False
            case "bishop" if abs(dx) != abs(dy):
                return False
            case "rook" | "bishop":
                sx, sy = (dx > 0) - (dx < 0), (dy > 0) - (dy < 0)
                cx, cy = self.x + sx, self.y + sy
                while (cx, cy) != (x, y):
                    if (cx, cy) in occupied:
                        return False
                    cx, cy = cx + sx, cy + sy
                return True
        raise ValueError(self.kind)


class TestChess(unittest.TestCase):
    def test_off_board(self):
        for kind in ("rook", "bishop", "knight"):
            p = Piece(kind, 0, 0)
            for x, y in [(-1, 0), (0, -1), (8, 0), (0, 8), (100, 100)]:
                self.assertFalse(p.can_move_to(x, y), (kind, x, y))

    def test_stay_in_place_is_invalid(self):
        self.assertFalse(Piece("rook", 3, 3).can_move_to(3, 3))

    def test_knight(self):
        n = Piece("knight", 4, 4)
        valid = {(x, y) for x in range(8) for y in range(8) if n.can_move_to(x, y)}
        self.assertEqual(len(valid), 8)
        self.assertEqual(len([1 for x in range(8) for y in range(8) if Piece("knight", 0, 0).can_move_to(x, y)]), 2)

    def test_rook_blocked(self):
        r = Piece("rook", 0, 0)
        self.assertTrue(r.can_move_to(0, 7))
        self.assertFalse(r.can_move_to(0, 7, occupied={(0, 3)}))
        self.assertFalse(r.can_move_to(1, 1))

    def test_bishop(self):
        b = Piece("bishop", 2, 2)
        self.assertTrue(b.can_move_to(7, 7) and b.can_move_to(0, 4))
        self.assertFalse(b.can_move_to(2, 5))


run(TestChess)

.

.

.

.

.

----------------------------------------------------------------------
Ran 5 tests in 0.001s

OK


## 11.4 No Test Tools
How would you load-test a webpage without testing tools? Identify the performance-critical criteria (response time, throughput, resource usage), then write your own driver with threads that simulate many concurrent users and record latencies. Below, the "server" is a local function with a simulated capacity limit.

In [5]:
import statistics
import threading
import time
from concurrent.futures import ThreadPoolExecutor

_capacity = threading.Semaphore(8)


def fake_endpoint() -> None:
    with _capacity:              # server can handle 8 concurrent requests
        time.sleep(0.005)


def load_test(users: int, requests_per_user: int) -> dict[str, float]:
    latencies: list[float] = []
    lock = threading.Lock()

    def user() -> None:
        for _ in range(requests_per_user):
            t = time.perf_counter()
            fake_endpoint()
            with lock:
                latencies.append(time.perf_counter() - t)

    start = time.perf_counter()
    with ThreadPoolExecutor(users) as ex:
        for _ in range(users):
            ex.submit(user)
    total = time.perf_counter() - start
    latencies.sort()
    return {
        "users": users,
        "throughput_rps": round(len(latencies) / total),
        "p50_ms": round(1000 * statistics.median(latencies), 1),
        "p95_ms": round(1000 * latencies[int(0.95 * len(latencies))], 1),
    }


for u in (1, 8, 32):
    print(load_test(u, 10))

{'users': 1, 'throughput_rps': 161, 'p50_ms': 6.3, 'p95_ms': 6.3}
{'users': 8, 'throughput_rps': 1240, 'p50_ms': 6.3, 'p95_ms': 6.7}


{'users': 32, 'throughput_rps': 1301, 'p50_ms': 6.3, 'p95_ms': 124.2}


## 11.5 Test a Pen
Ask first: who uses it (children? artists?), and for what (drawing, writing on fabric)? Then test that it **fulfils its intended use**, plus safety, durability, and edge conditions. The same approach works in code: turn the requirements into a spec, then write tests against that spec.

In [6]:
@dataclass
class Pen:
    ink_ml: float = 1.0
    color: str = "blue"
    washable: bool = True
    ml_per_meter: float = 0.01

    def write(self, meters: float, surface: str) -> float:
        if surface not in {"paper", "fabric", "skin"}:
            raise ValueError("unsupported surface")
        written = min(meters, self.ink_ml / self.ml_per_meter)
        self.ink_ml -= written * self.ml_per_meter
        return written


class TestPenForKids(unittest.TestCase):
    """Spec (after asking): children's washable marker, writes on paper/fabric/skin, ~100m of ink."""

    def test_intended_use(self):
        self.assertEqual(Pen().write(1, "paper"), 1)

    def test_runs_out_gracefully(self):
        p = Pen()
        self.assertAlmostEqual(p.write(1000, "paper"), 100)
        self.assertEqual(p.write(1, "paper"), 0)

    def test_safety_washable(self):
        self.assertTrue(Pen().washable)

    def test_bad_surface(self):
        with self.assertRaises(ValueError):
            Pen().write(1, "glass")


run(TestPenForKids)

.

.

.

.

----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


## 11.6 Test an ATM
How would you test an ATM in a distributed banking system? Consider the users, the use cases (withdraw, deposit, balance), concurrency (two ATMs on one account), network failures, and security. Key invariant: **money is never created or lost**, even under concurrency and partial failure.

In [7]:
class Bank:
    def __init__(self) -> None:
        self.balances: dict[str, int] = {}
        self._lock = threading.Lock()

    def withdraw(self, acct: str, amount: int) -> bool:
        if amount <= 0:
            raise ValueError("amount must be positive")
        with self._lock:  # remove this lock and test_concurrent_withdrawals can fail
            if self.balances.get(acct, 0) < amount:
                return False
            bal = self.balances[acct]
            time.sleep(0)  # widen the race window
            self.balances[acct] = bal - amount
            return True


class ATM:
    def __init__(self, bank: Bank, cash: int) -> None:
        self.bank, self.cash = bank, cash

    def withdraw(self, acct: str, amount: int) -> bool:
        if amount > self.cash:
            return False                     # check machine cash first so nothing is debited
        if not self.bank.withdraw(acct, amount):
            return False
        self.cash -= amount
        return True


class TestATM(unittest.TestCase):
    def setUp(self):
        self.bank = Bank()
        self.bank.balances["alice"] = 100

    def test_normal(self):
        self.assertTrue(ATM(self.bank, 500).withdraw("alice", 40))
        self.assertEqual(self.bank.balances["alice"], 60)

    def test_insufficient_funds(self):
        self.assertFalse(ATM(self.bank, 500).withdraw("alice", 101))

    def test_atm_out_of_cash_does_not_debit(self):
        self.assertFalse(ATM(self.bank, 10).withdraw("alice", 50))
        self.assertEqual(self.bank.balances["alice"], 100)

    def test_invalid_amount(self):
        with self.assertRaises(ValueError):
            ATM(self.bank, 500).withdraw("alice", -5)

    def test_concurrent_withdrawals(self):
        self.bank.balances["alice"] = 1000
        atms = [ATM(self.bank, 10_000) for _ in range(4)]
        with ThreadPoolExecutor(8) as ex:
            results = list(ex.map(lambda i: atms[i % 4].withdraw("alice", 1), range(2000)))
        self.assertEqual(sum(results), 1000)
        self.assertEqual(self.bank.balances["alice"], 0)
        self.assertEqual(sum(10_000 - a.cash for a in atms), 1000)  # conservation of money


run(TestATM)

.

.

.

.

.

----------------------------------------------------------------------
Ran 5 tests in 0.028s

OK
